# 🚢 Titanic Survival Analysis

Full EDA on the Titanic dataset (891 passengers).

**What this notebook covers:**
1. Load & preview the data
2. Clean missing values (Age, Cabin, Embarked)
3. Feature engineering (Title, FamilySize, AgeGroup)
4. Descriptive statistics
5. 14 interactive charts
6. AI insights from Claude

---
**Install requirements:**
```
pip install pandas numpy plotly anthropic openpyxl scipy statsmodels
```
**Set API key:**
```
export ANTHROPIC_API_KEY="sk-ant-..."
```

---
## Step 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import anthropic

print('Libraries imported ✅')

---
## Step 2 — Load the Dataset

In [ ]:
# Load the file — change this path if needed
df = pd.read_csv('titanic.csv')

print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

---
## Step 3 — Preview & Missing Values

In [ ]:
# First 5 rows
df.head()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print()
# Age: 177 missing
# Cabin: 687 missing (77%!) — too many to impute
# Embarked: 2 missing — easy to fill

---
## Step 4 — Clean the Data

In [ ]:
# Save missing counts before cleaning (used in the AI summary later)
missing_before = df.isnull().sum()[df.isnull().sum() > 0].to_dict()

# ── Fix Embarked: only 2 missing, fill with most common port 'S' ─
df['Embarked'] = df['Embarked'].fillna('S')

# ── Fix Cabin: 77% missing — too many to impute, label as Unknown ─
df['Cabin'] = df['Cabin'].fillna('Unknown')

# ── Fix Age: fill with median of the SAME Pclass + Sex group ─────
# This is smarter than a global median.
# A 1st-class adult male is probably older than a 3rd-class female.
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
# Safety net: if any Age is still NaN, use the global median
df['Age'] = df['Age'].fillna(df['Age'].median())

print('After cleaning:')
print(df.isnull().sum())

---
## Step 5 — Feature Engineering

We create new columns that make analysis more meaningful.

In [ ]:
# ── Title: extract from Name ──────────────────────────────────
# Name looks like: 'Braund, Mr. Owen Harris'
# We grab the word between ', ' and '.'
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')

# Group uncommon titles as 'Rare' to keep charts clean
rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Mlle', 'Countess',
               'Ms', 'Lady', 'Jonkheer', 'Don', 'Dona', 'Capt', 'Sir']
df['Title'] = df['Title'].apply(lambda t: 'Rare' if t in rare_titles else t)

print('Titles found:', df['Title'].value_counts().to_dict())

In [ ]:
# ── FamilySize: SibSp + Parch + 1 (the passenger themselves) ─
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# ── IsAlone: 1 if travelling alone ───────────────────────────
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# ── AgeGroup: bucket ages into readable labels ────────────────
df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child (0-12)', 'Teen (13-18)', 'Adult (19-35)',
            'Middle-aged (36-60)', 'Senior (60+)']
)

# ── Human-readable labels ─────────────────────────────────────
df['PclassLabel']   = df['Pclass'].map({1: '1st Class', 2: '2nd Class', 3: '3rd Class'})
df['SurvivedLabel'] = df['Survived'].map({0: 'Did Not Survive', 1: 'Survived'})

print('New columns created: Title, FamilySize, IsAlone, AgeGroup, PclassLabel, SurvivedLabel')
df[['Name', 'Title', 'FamilySize', 'IsAlone', 'AgeGroup', 'SurvivedLabel']].head()

---
## Step 6 — Descriptive Statistics

In [ ]:
# Key statistics
print(f"Overall survival rate : {df['Survived'].mean()*100:.1f}%")
print(f"Female survival rate  : {df[df['Sex']=='female']['Survived'].mean()*100:.1f}%")
print(f"Male survival rate    : {df[df['Sex']=='male']['Survived'].mean()*100:.1f}%")
print()
print('Survival rate by class:')
print(df.groupby('PclassLabel')['Survived'].mean().mul(100).round(1).to_string())
print()
print(f"Travelling alone      : {df[df['IsAlone']==1]['Survived'].mean()*100:.1f}% survived")
print(f"Travelling with family: {df[df['IsAlone']==0]['Survived'].mean()*100:.1f}% survived")
print()
print(f"Children (age<=12)    : {df[df['Age']<=12]['Survived'].mean()*100:.1f}% survived")
print(f"Median age            : {df['Age'].median():.0f} years")
print(f"Median fare           : {df['Fare'].median():.2f}")

In [ ]:
# Full descriptive statistics for numeric columns
df[['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']].describe().round(2)

---
## Step 7 — Visualizations (14 charts)

All charts are interactive — hover, zoom, click legends.

In [ ]:
# Chart 1: Overall survival count
counts = df['SurvivedLabel'].value_counts().reset_index()
counts.columns = ['Status', 'Count']
fig = px.bar(
    counts, x='Status', y='Count',
    title='1. Overall Survival Count',
    color='Status',
    color_discrete_map={'Survived': '#34d399', 'Did Not Survive': '#f87171'}
)
fig.show()

In [ ]:
# Chart 2: Survival rate by passenger class
surv_class = df.groupby('PclassLabel')['Survived'].mean().reset_index()
surv_class['Survival Rate (%)'] = (surv_class['Survived'] * 100).round(1)
fig = px.bar(
    surv_class, x='PclassLabel', y='Survival Rate (%)',
    title='2. Survival Rate by Passenger Class',
    color='PclassLabel',
    color_discrete_sequence=['#6366f1', '#a78bfa', '#c4b5fd'],
    text='Survival Rate (%)'
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

In [ ]:
# Chart 3: Survival rate by sex
surv_sex = df.groupby('Sex')['Survived'].mean().reset_index()
surv_sex['Survival Rate (%)'] = (surv_sex['Survived'] * 100).round(1)
fig = px.bar(
    surv_sex, x='Sex', y='Survival Rate (%)',
    title='3. Survival Rate by Sex',
    color='Sex',
    color_discrete_map={'male': '#60a5fa', 'female': '#f472b6'},
    text='Survival Rate (%)'
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

In [ ]:
# Chart 4: Survival rate by class AND sex (grouped bars)
surv_cs = df.groupby(['PclassLabel', 'Sex'])['Survived'].mean().reset_index()
surv_cs['Survival Rate (%)'] = (surv_cs['Survived'] * 100).round(1)
fig = px.bar(
    surv_cs, x='PclassLabel', y='Survival Rate (%)',
    color='Sex', barmode='group',
    title='4. Survival Rate by Class and Sex',
    color_discrete_map={'male': '#60a5fa', 'female': '#f472b6'},
    text='Survival Rate (%)'
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

In [ ]:
# Chart 5: Age distribution by survival
fig = px.histogram(
    df, x='Age', color='SurvivedLabel',
    title='5. Age Distribution by Survival',
    nbins=30, barmode='overlay', opacity=0.75,
    color_discrete_map={'Survived': '#34d399', 'Did Not Survive': '#f87171'}
)
fig.show()

In [ ]:
# Chart 6: Fare distribution by survival (capped at 300 to avoid outlier squish)
fig = px.histogram(
    df[df['Fare'] < 300], x='Fare', color='SurvivedLabel',
    title='6. Fare Distribution by Survival (capped at 300)',
    nbins=40, barmode='overlay', opacity=0.75,
    color_discrete_map={'Survived': '#34d399', 'Did Not Survive': '#f87171'}
)
fig.show()

In [ ]:
# Chart 7: Pie chart — passenger split by sex
fig = px.pie(
    df, names='Sex',
    title='7. Passenger Split by Sex',
    color='Sex',
    color_discrete_map={'male': '#60a5fa', 'female': '#f472b6'}
)
fig.show()

In [ ]:
# Chart 8: Pie chart — embarkation port
fig = px.pie(
    df, names='Embarked',
    title='8. Passengers by Embarkation Port (C=Cherbourg, Q=Queenstown, S=Southampton)',
    color_discrete_sequence=['#6366f1', '#f472b6', '#34d399']
)
fig.show()

In [ ]:
# Chart 9: Survival rate by age group
surv_age = df.groupby('AgeGroup', observed=True)['Survived'].mean().reset_index()
surv_age['Survival Rate (%)'] = (surv_age['Survived'] * 100).round(1)
fig = px.bar(
    surv_age, x='AgeGroup', y='Survival Rate (%)',
    title='9. Survival Rate by Age Group',
    color='AgeGroup', text='Survival Rate (%)'
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

In [ ]:
# Chart 10: Box plot — fare by passenger class
fig = px.box(
    df[df['Fare'] < 300], x='PclassLabel', y='Fare',
    title='10. Fare Distribution by Passenger Class',
    color='PclassLabel',
    color_discrete_sequence=['#6366f1', '#a78bfa', '#c4b5fd']
)
fig.show()

In [ ]:
# Chart 11: Survival rate by family size (line chart)
surv_fam = df.groupby('FamilySize')['Survived'].mean().reset_index()
surv_fam['Survival Rate (%)'] = (surv_fam['Survived'] * 100).round(1)
fig = px.line(
    surv_fam, x='FamilySize', y='Survival Rate (%)',
    title='11. Survival Rate by Family Size',
    markers=True, color_discrete_sequence=['#f472b6']
)
fig.update_traces(line_width=3, marker_size=8)
fig.show()

In [ ]:
# Chart 12: Survival rate by title
surv_title = df.groupby('Title')['Survived'].mean().reset_index()
surv_title['Survival Rate (%)'] = (surv_title['Survived'] * 100).round(1)
surv_title = surv_title.sort_values('Survival Rate (%)', ascending=False)
fig = px.bar(
    surv_title, x='Title', y='Survival Rate (%)',
    title='12. Survival Rate by Title',
    color='Survival Rate (%)', color_continuous_scale='RdYlGn',
    text='Survival Rate (%)'
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

In [ ]:
# Chart 13: Correlation heatmap
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
corr = df[numeric_cols].corr().round(2)
fig = px.imshow(
    corr, text_auto=True,
    title='13. Correlation Heatmap',
    color_continuous_scale='RdBu_r', aspect='auto'
)
fig.show()

In [ ]:
# Chart 14: Scatter — Age vs Fare, coloured by survival
fig = px.scatter(
    df[df['Fare'] < 300], x='Age', y='Fare',
    color='SurvivedLabel',
    title='14. Age vs Fare (coloured by Survival)',
    color_discrete_map={'Survived': '#34d399', 'Did Not Survive': '#f87171'},
    opacity=0.6
)
fig.show()

---
## Step 8 — AI Insights with Claude

> Make sure `ANTHROPIC_API_KEY` is set before running this cell.

In [ ]:
# Build a summary of key stats to send to Claude
survival_rate = df['Survived'].mean() * 100
female_surv   = df[df['Sex'] == 'female']['Survived'].mean() * 100
male_surv     = df[df['Sex'] == 'male']['Survived'].mean() * 100
class1_surv   = df[df['Pclass'] == 1]['Survived'].mean() * 100
class2_surv   = df[df['Pclass'] == 2]['Survived'].mean() * 100
class3_surv   = df[df['Pclass'] == 3]['Survived'].mean() * 100
alone_surv    = df[df['IsAlone'] == 1]['Survived'].mean() * 100
family_surv   = df[df['IsAlone'] == 0]['Survived'].mean() * 100
child_surv    = df[df['Age'] <= 12]['Survived'].mean() * 100

summary = f"""
Titanic Dataset (891 passengers):

Overall survival rate: {survival_rate:.1f}%
Female survival rate: {female_surv:.1f}%
Male survival rate: {male_surv:.1f}%

By class:
  1st Class: {class1_surv:.1f}%
  2nd Class: {class2_surv:.1f}%
  3rd Class: {class3_surv:.1f}%

Children (age 12 and under): {child_surv:.1f}% survived
Median age: {df['Age'].median():.0f} years
Median fare: {df['Fare'].median():.2f}

Travelling alone: {alone_surv:.1f}% survived
Travelling with family: {family_surv:.1f}% survived
"""

print(summary)

In [ ]:
# Send to Claude and get insights
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment

prompt = f"""
You are a senior data analyst presenting findings about the Titanic disaster.
Here is the statistical summary:

{summary}

Please write a clear analysis covering:
1. A brief overview (2-3 sentences)
2. The most striking survival patterns
3. Key correlations and what they mean
4. What this tells us about social inequality in 1912
5. A memorable conclusion

Be specific. Use the exact numbers. Write for a general audience.
"""

message = client.messages.create(
    model='claude-opus-4-5',
    max_tokens=1024,
    messages=[{'role': 'user', 'content': prompt}]
)

insights = message.content[0].text

print('=' * 60)
print('AI INSIGHTS')
print('=' * 60)
print(insights)

---
## ✅ Analysis Complete!

You've covered:
- Data cleaning (Age, Cabin, Embarked)
- Feature engineering (Title, FamilySize, AgeGroup)
- 14 interactive Plotly charts
- AI-generated insights from Claude

**Key findings to highlight in your report:**
- Women survived at a far higher rate than men
- 1st class passengers had dramatically better survival odds
- Children had a higher survival rate than adults
- Travelling with a small family slightly improved survival odds
- Higher fare paid strongly correlated with survival